In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from processing import *
from wavelengths import *
from fit_prefilter import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'

folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2025-09-16/'
#folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2024-07-10/'
files = sorted(glob.glob(folder + '*.fits.gz'))

#for file in files:
#    process(file, dark_file=dark_file,
#            _calc_wavelengths=True, _find_center=True, _mask=True, _rebin=8, to_file=True)

In [3]:
files = sorted(glob.glob('*.fits'))

In [75]:
k_T = 0.032
q_V = 299792458 / 6173.341

wvlns = []
temperatures = []
velocities = []
datas = []
mus = []

for file in files:
    with fits.open(file) as hdul:
        data = hdul[0].data
        header = hdul[0].header

    nx, ny = header['NAXIS2'], header['NAXIS1']
    xc, yc = header['CRPIX2'] - 1, header['CRPIX1'] - 1
    rsun = header['RSUN_ARC'] / header['CDELT1']

    xi, yi = np.mgrid[:nx,:ny]
    mu = np.sqrt((rsun ** 2 - (xi - xc) ** 2 - (yi - yc) ** 2).clip(0)) / rsun

    datas += [data]
    wvlns += [read_wavelengths(header)]
    temperatures += [header['FGOV1PT1']]
    velocities += [header['OBS_VR']]
    mus += [mu]

datas = np.swapaxes(np.concat(datas, axis=1), 0, 1)
wvlns = np.array(wvlns)
temperatures = np.array(temperatures)
velocities = np.array(velocities)
mus = np.array(mus)
mu_ = np.nanmean(mus, axis=0)
voltages = (wvlns - 6173.341 - (np.expand_dims(temperatures, -1) - 61) * 4.01225e-2) / 3.513e-4

wvlns -= k_T * (np.expand_dims(temperatures, -1) - 61)

print(temperatures, velocities)

[61.   66.   55.99] [ -69.95156039  805.7347685  1305.12722331]


In [79]:
x, y = 50, 200

plt.figure(figsize=(10,8))
plt.plot(voltages[0], np.nanmedian(datas[0,:], axis=(-1,-2)))
plt.plot(voltages[1], np.nanmedian(datas[1,:], axis=(-1,-2)))
plt.plot(voltages[2], np.nanmedian(datas[2,:], axis=(-1,-2)))
plt.tight_layout()

In [80]:
500 * 0.15e-4

0.007500000000000001

In [81]:
dwv = 0.01

a, b = np.max(np.min(wvlns, axis=-1)), np.min(np.max(wvlns, axis=-1))
a = np.round(a + dwv / 2, 2)
b = np.round(b - dwv / 2, 2)

wv = np.arange(a, b + dwv / 2, dwv)

In [82]:
from fitting import interpolate

Q = []
for i in range(3):
    Q += [interpolate(datas[i], wvlns[i], np.expand_dims(wv, (-1,-2)))]
Q = np.array(Q)

In [83]:
x, y = 50, 200

plt.figure(figsize=(10,8))
for i in range(3):
    plt.plot(wv, Q[i,:,x,y])
plt.tight_layout()

In [84]:
params, gamma, res = fit_prefilter(*Q, wv,
                                   axis=0, lam=0, niter=10)

print(gamma, np.nanmedian(res))

[ 0.0536992  -0.14825684  0.18284304] 82.26310979677439


In [85]:
delta1, delta2 = gamma[-2:]

delta1 += k_T * (temperatures[1] - temperatures[0])
delta2 += k_T * (temperatures[2] - temperatures[0])

delta1_ = (velocities[1] - velocities[0]) / q_V
delta2_ = (velocities[2] - velocities[0]) / q_V

delta1, delta2, delta1_, delta2_

(np.float64(0.011743156027238189),
 np.float64(0.022523040797329447),
 np.float64(0.018032175837044428),
 np.float64(0.028315689761670072))

In [50]:
params1 = np.expand_dims(params, 1)
params2 = params1.copy()
params3 = params1.copy()

params2[0] += gamma[-2]
params3[0] += gamma[-1]

Q0 = datas[0] / voigt_func((np.expand_dims(wvlns[0], (1,2)), 0), *params1, *gamma)[0]
Q1 = datas[1] / voigt_func((np.expand_dims(wvlns[1], (1,2)), 0), *params2, *gamma)[0]
Q2 = datas[2] / voigt_func((np.expand_dims(wvlns[2], (1,2)), 0), *params3, *gamma)[0]

In [51]:
x, y = 50, 200
#x, y = 70, 60
#x, y = 120, 130

plt.figure(figsize=(10,8))
plt.plot(wvlns[0], datas[0,:,x,y])
plt.plot(wvlns[1], datas[1,:,x,y])
plt.plot(wvlns[2], datas[2,:,x,y])

plt.plot(wvlns[0], Q0[:,x,y])
plt.plot(wvlns[1], Q1[:,x,y])
plt.plot(wvlns[2], Q2[:,x,y])
plt.tight_layout()

In [52]:
plt.figure(figsize=(10,10))
plt.imshow(params[0], 'seismic', vmin=np.min(wv), vmax=np.max(wv))
plt.tight_layout()

In [53]:
plt.figure(figsize=(10,10))
plt.imshow(-params[1], 'inferno', vmin=0.05, vmax=0.08)
plt.tight_layout()

In [54]:
plt.figure(figsize=(10,8))
plt.plot(mu_.flatten(), -params[1].flatten(), '.', ms=0.5)

plt.ylim(0.04, 0.1)
plt.tight_layout()

In [55]:
plt.figure(figsize=(10,10))
plt.imshow(params[2], 'inferno', vmin=0.035, vmax=0.06)
plt.tight_layout()

In [57]:
plt.figure(figsize=(10,8))
plt.plot(mu_.flatten(), params[2].flatten() * 2 * np.sqrt(2 * np.log(2)), '.', ms=0.5)

plt.ylim(0.0, 0.2)
plt.tight_layout()

In [19]:
plt.figure(figsize=(10,10))
plt.imshow(res, 'inferno', vmin=0, vmax=200)
plt.tight_layout()

In [20]:
plt.figure(figsize=(10,10))
plt.imshow(Q1[-1] / Q1[0], 'inferno', vmin=0.8, vmax=1.05)
plt.tight_layout()